# Houston SAR Flood Mapping - Local ML Workflow

This notebook is a clean local version of the Houston flood-classification workflow. It does not use Google Earth Engine, Google Drive, or interactive map widgets.

Inputs stay in the downloaded folder. The notebook only reads them from there and writes derived outputs to an `outputs_local` folder beside those downloaded inputs.

Models compared here: Random Forest, optional XGBoost, and optional 1D CNN.

## 1. Dependency Check

Run this first. It installs missing core packages into the active notebook kernel. TensorFlow is included so the CNN section can run. XGBoost is optional and is used only if it is already available.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "rasterio": "rasterio",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
    "tensorflow": "tensorflow",
}

missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Core packages are available.")

print("Python:", sys.executable)

Core packages are available.
Python: /home/abdullah/miniconda3/envs/landlab_debrisflow/bin/python


## 2. Setup

The input paths below point directly to the downloaded files. No input data is copied into this repository.

In [2]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except Exception as exc:
    xgb = None
    HAS_XGBOOST = False
    print("XGBoost not available. Random Forest will still run.")

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TENSORFLOW = True
except Exception as exc:
    tf = None
    keras = None
    layers = None
    HAS_TENSORFLOW = False
    print("TensorFlow not available. CNN will be skipped until TensorFlow is installed.")

warnings.filterwarnings("ignore", category=UserWarning)

INPUT_DIR = Path("/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M")
IMAGE_PATH = INPUT_DIR / "Houston_DL_Sent-1.tif"
SAMPLE_PATH = INPUT_DIR / "Houston_fl_samples.csv"
OUTPUT_DIR = INPUT_DIR / "outputs_local"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["VV", "VV_1", "VH", "VH_1"]
LABEL = "classvalue"
SPLIT = "sample"

# Based on the original notebook. Change this if your label definition says otherwise.
FLOOD_CLASS = 1

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
if HAS_TENSORFLOW:
    tf.random.set_seed(RANDOM_STATE)

for path in [IMAGE_PATH, SAMPLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Image:", IMAGE_PATH)
print("Samples:", SAMPLE_PATH)
print("Outputs:", OUTPUT_DIR)
print("XGBoost available:", HAS_XGBOOST)
print("TensorFlow/CNN available:", HAS_TENSORFLOW)

XGBoost not available. Random Forest will still run.


I0000 00:00:1780377977.516403    2114 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780377977.519379    2114 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780377977.873431    2114 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780377979.532845    2114 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Image: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/Houston_DL_Sent-1.tif
Samples: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/Houston_fl_samples.csv
Outputs: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local
XGBoost available: False
TensorFlow/CNN available: True


## 3. Inspect Inputs

In [3]:
samples = pd.read_csv(SAMPLE_PATH)

required_columns = FEATURES + [LABEL, SPLIT]
missing_columns = [column for column in required_columns if column not in samples.columns]
if missing_columns:
    raise ValueError(f"Missing required sample columns: {missing_columns}")

samples = samples.dropna(subset=required_columns).copy()
samples[LABEL] = samples[LABEL].astype(int)

print("Sample table shape:", samples.shape)
display(pd.crosstab(samples[SPLIT], samples[LABEL], margins=True))
display(samples[FEATURES].describe().T)

with rasterio.open(IMAGE_PATH) as src:
    print("Raster width, height:", src.width, src.height)
    print("Band count:", src.count)
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("Band descriptions:", src.descriptions)
    print("Dtype:", src.dtypes)
    print("Nodata:", src.nodata)

if len(FEATURES) != 4:
    raise ValueError("This notebook expects four Sentinel-1 feature bands.")

Sample table shape: (3877, 6)


classvalue,1,2,All
sample,,,
test,2612,538,3150
train,521,206,727
All,3133,744,3877


,count,mean,std,min,25%,50%,75%,max
VV,3877.0,-12.682193,3.782759,-20.679399,-15.332097,-13.423549,-10.553363,10.932773
VV_1,3877.0,1.160397,0.534703,-10.723483,0.955610,1.136658,1.349888,12.705044
VH,3877.0,1.169416,0.254022,0.466522,0.994604,1.151373,1.318105,2.723428
VH_1,3877.0,-20.846086,4.983147,-34.962729,-24.685985,-21.570793,-16.678534,-6.296876


Raster width, height: 557 338
Band count: 4
CRS: EPSG:4326
Transform: | 0.00, 0.00,-95.75|
| 0.00,-0.00, 29.79|
| 0.00, 0.00, 1.00|
Band descriptions: ('VV', 'VV_1', 'VH', 'VH_1')
Dtype: ('float32', 'float32', 'float32', 'float32')
Nodata: None


## 4. Train/Test Split

The CSV already contains a `sample` column. This cell preserves that split instead of making a new random split.

In [4]:
train = samples[samples[SPLIT].str.lower() == "train"].copy()
test = samples[samples[SPLIT].str.lower() == "test"].copy()

if train.empty or test.empty:
    raise ValueError("The sample column must contain both 'train' and 'test' rows.")

X_train = train[FEATURES].to_numpy(dtype="float32")
y_train = train[LABEL].to_numpy(dtype="int32")
X_test = test[FEATURES].to_numpy(dtype="float32")
y_test = test[LABEL].to_numpy(dtype="int32")

classes = sorted(samples[LABEL].unique().tolist())
label_to_index = {label: idx for idx, label in enumerate(classes)}
index_to_label = {idx: label for label, idx in label_to_index.items()}
n_classes = len(classes)
y_train_index = np.array([label_to_index[label] for label in y_train], dtype="int32")
y_test_index = np.array([label_to_index[label] for label in y_test], dtype="int32")

if FLOOD_CLASS not in classes:
    raise ValueError(f"FLOOD_CLASS={FLOOD_CLASS} is not present in labels {classes}")

print("Classes:", classes)
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Classes: [1, 2]
Train shape: (727, 4) (727,)
Test shape: (3150, 4) (3150,)


## 5. Train Models

Random Forest is the project baseline, XGBoost is an optional tree-based comparison, and the 1D CNN reproduces the neural-network direction from the original notebook. The CNN uses standardized four-band pixel features, class weighting, and early stopping.

In [5]:
models = {}

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
models["random_forest"] = {
    "model": rf_model,
    "kind": "sklearn_labels",
}

if HAS_XGBOOST:
    xgb_model = xgb.XGBClassifier(
        objective="binary:logistic" if len(classes) == 2 else "multi:softprob",
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss" if len(classes) == 2 else "mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_model.fit(X_train, y_train_index)
    models["xgboost"] = {
        "model": xgb_model,
        "kind": "indexed_labels",
    }

if HAS_TENSORFLOW:
    cnn_scaler = StandardScaler()
    X_train_cnn = cnn_scaler.fit_transform(X_train).astype("float32").reshape(-1, len(FEATURES), 1)
    X_test_cnn = cnn_scaler.transform(X_test).astype("float32").reshape(-1, len(FEATURES), 1)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(n_classes),
        y=y_train_index,
    )
    class_weight = {idx: float(weight) for idx, weight in enumerate(class_weights)}

    cnn_model = keras.Sequential([
        layers.Input(shape=(len(FEATURES), 1)),
        layers.Conv1D(32, kernel_size=2, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv1D(64, kernel_size=2, padding="same", activation="relu"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(n_classes, activation="softmax"),
    ])
    cnn_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5),
    ]

    cnn_history = cnn_model.fit(
        X_train_cnn,
        y_train_index,
        validation_data=(X_test_cnn, y_test_index),
        epochs=200,
        batch_size=64,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0,
    )

    cnn_history_path = OUTPUT_DIR / "cnn_1d_training_history.csv"
    pd.DataFrame(cnn_history.history).to_csv(cnn_history_path, index_label="epoch")

    models["cnn_1d"] = {
        "model": cnn_model,
        "kind": "cnn_1d",
        "scaler": cnn_scaler,
        "history_path": cnn_history_path,
    }

print("Trained models:", list(models.keys()))

E0000 00:00:1780377999.346440    2114 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1780377999.346764    2671 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1780377999.370025    2114 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Trained models: ['random_forest', 'cnn_1d']


## 6. Evaluate Models

In [6]:
def predict_labels_and_flood_probability(model_info, X):
    model = model_info["model"]
    kind = model_info["kind"]

    if kind == "sklearn_labels":
        labels = model.predict(X).astype("int32")
        probabilities = model.predict_proba(X)
        class_list = list(model.classes_)
        flood_col = class_list.index(FLOOD_CLASS)
        flood_probability = probabilities[:, flood_col]
        return labels, flood_probability

    if kind == "indexed_labels":
        pred_index = model.predict(X).astype("int32")
        labels = np.array([index_to_label[int(idx)] for idx in pred_index], dtype="int32")
        probabilities = model.predict_proba(X)
        flood_col = label_to_index[FLOOD_CLASS]
        flood_probability = probabilities[:, flood_col]
        return labels, flood_probability

    if kind == "cnn_1d":
        scaler = model_info["scaler"]
        X_cnn = scaler.transform(X).astype("float32").reshape(-1, len(FEATURES), 1)
        probabilities = model.predict(X_cnn, batch_size=4096, verbose=0)
        pred_index = np.argmax(probabilities, axis=1).astype("int32")
        labels = np.array([index_to_label[int(idx)] for idx in pred_index], dtype="int32")
        flood_col = label_to_index[FLOOD_CLASS]
        flood_probability = probabilities[:, flood_col]
        return labels, flood_probability

    raise ValueError(f"Unknown model kind: {kind}")


evaluation_rows = []
reports = {}

for model_name, model_info in models.items():
    y_pred, flood_probability = predict_labels_and_flood_probability(model_info, X_test)
    report = classification_report(y_test, y_pred, labels=classes, output_dict=True, zero_division=0)
    reports[model_name] = report

    flood_report = report[str(FLOOD_CLASS)]
    evaluation_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": report["macro avg"]["f1-score"],
        "flood_precision": flood_report["precision"],
        "flood_recall": flood_report["recall"],
        "flood_f1": flood_report["f1-score"],
    })

metrics = pd.DataFrame(evaluation_rows).sort_values(["flood_f1", "macro_f1"], ascending=False)
metrics_path = OUTPUT_DIR / "model_metrics.csv"
reports_path = OUTPUT_DIR / "classification_reports.json"

metrics.to_csv(metrics_path, index=False)
with reports_path.open("w", encoding="utf-8") as f:
    json.dump(reports, f, indent=2)

display(metrics)
print("Saved metrics:", metrics_path)
print("Saved reports:", reports_path)

best_model_name = metrics.iloc[0]["model"]
best_model_info = models[best_model_name]
print("Best model by flood F1:", best_model_name)

,model,accuracy,macro_f1,flood_precision,flood_recall,flood_f1
0,random_forest,0.798095,0.722259,0.952381,0.796325,0.867389
1,cnn_1d,0.728571,0.665138,0.960189,0.701761,0.810883


Saved metrics: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/model_metrics.csv
Saved reports: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/classification_reports.json
Best model by flood F1: random_forest


## 7. Confusion Matrix For Best Model

In [7]:
best_pred, _ = predict_labels_and_flood_probability(best_model_info, X_test)
cm = confusion_matrix(y_test, best_pred, labels=classes)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])
cm_path = OUTPUT_DIR / f"{best_model_name}_confusion_matrix.csv"
cm_df.to_csv(cm_path)

display(cm_df)
print(classification_report(y_test, best_pred, labels=classes, zero_division=0))
print("Saved confusion matrix:", cm_path)

,pred_1,pred_2
true_1,2080,532
true_2,104,434


              precision    recall  f1-score   support

           1       0.95      0.80      0.87      2612
           2       0.45      0.81      0.58       538

    accuracy                           0.80      3150
   macro avg       0.70      0.80      0.72      3150
weighted avg       0.87      0.80      0.82      3150

Saved confusion matrix: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_confusion_matrix.csv


## 8. Predict The Full Raster

This creates two GeoTIFFs:

- class map, with labels from the training data and nodata value `0`
- flood probability map for `FLOOD_CLASS`, with nodata value `-9999`

In [8]:
BATCH_SIZE = 65536

with rasterio.open(IMAGE_PATH) as src:
    profile = src.profile.copy()
    image = src.read(list(range(1, len(FEATURES) + 1))).astype("float32")

bands, height, width = image.shape
pixels = np.moveaxis(image, 0, -1).reshape(-1, bands)
valid = np.all(np.isfinite(pixels), axis=1)
valid_indices = np.flatnonzero(valid)

class_flat = np.zeros(pixels.shape[0], dtype="uint8")
flood_probability_flat = np.full(pixels.shape[0], np.nan, dtype="float32")

for start in range(0, len(valid_indices), BATCH_SIZE):
    batch_indices = valid_indices[start:start + BATCH_SIZE]
    labels, flood_probability = predict_labels_and_flood_probability(best_model_info, pixels[batch_indices])
    class_flat[batch_indices] = labels.astype("uint8")
    flood_probability_flat[batch_indices] = flood_probability.astype("float32")

class_map = class_flat.reshape(height, width)
flood_probability_map = flood_probability_flat.reshape(height, width)

class_path = OUTPUT_DIR / f"{best_model_name}_class_map.tif"
probability_path = OUTPUT_DIR / f"{best_model_name}_flood_probability.tif"

class_profile = profile.copy()
class_profile.update(count=1, dtype="uint8", nodata=0, compress="lzw")
with rasterio.open(class_path, "w", **class_profile) as dst:
    dst.write(class_map, 1)

probability_profile = profile.copy()
probability_profile.update(count=1, dtype="float32", nodata=-9999.0, compress="lzw")
probability_to_write = np.where(np.isfinite(flood_probability_map), flood_probability_map, -9999.0).astype("float32")
with rasterio.open(probability_path, "w", **probability_profile) as dst:
    dst.write(probability_to_write, 1)

print("Valid pixels predicted:", int(valid.sum()))
print("Saved class map:", class_path)
print("Saved flood probability map:", probability_path)

Valid pixels predicted: 188266
Saved class map: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_class_map.tif
Saved flood probability map: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_flood_probability.tif


## 9. Save Model And Run Metadata

In [9]:
metadata_path = OUTPUT_DIR / f"{best_model_name}_run_metadata.json"
model_artifacts = {}

for model_name, model_info in models.items():
    kind = model_info["kind"]

    if kind == "cnn_1d":
        keras_path = OUTPUT_DIR / f"{model_name}_model.keras"
        scaler_path = OUTPUT_DIR / f"{model_name}_scaler.joblib"
        model_info["model"].save(keras_path)
        joblib.dump(model_info["scaler"], scaler_path)

        model_artifacts[model_name] = {
            "kind": kind,
            "model_file": str(keras_path),
            "scaler_file": str(scaler_path),
            "history_csv": str(model_info.get("history_path", "")),
        }
    else:
        model_path = OUTPUT_DIR / f"{model_name}_model.joblib"
        model_bundle = {
            "model_name": model_name,
            "kind": kind,
            "model": model_info["model"],
            "features": FEATURES,
            "label": LABEL,
            "classes": classes,
            "flood_class": FLOOD_CLASS,
            "label_to_index": label_to_index,
            "index_to_label": index_to_label,
        }
        joblib.dump(model_bundle, model_path)

        model_artifacts[model_name] = {
            "kind": kind,
            "model_file": str(model_path),
        }

artifacts_path = OUTPUT_DIR / "model_artifacts.json"
with artifacts_path.open("w", encoding="utf-8") as f:
    json.dump(model_artifacts, f, indent=2)

metadata = {
    "input_image": str(IMAGE_PATH),
    "input_samples": str(SAMPLE_PATH),
    "output_dir": str(OUTPUT_DIR),
    "features": FEATURES,
    "label": LABEL,
    "split_column": SPLIT,
    "classes": classes,
    "flood_class": FLOOD_CLASS,
    "best_model": best_model_name,
    "metrics_csv": str(metrics_path),
    "classification_reports_json": str(reports_path),
    "class_map": str(class_path),
    "flood_probability_map": str(probability_path),
    "model_artifacts_json": str(artifacts_path),
    "best_model_artifacts": model_artifacts[best_model_name],
}
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved model artifacts:", artifacts_path)
print("Saved metadata:", metadata_path)

Saved model artifacts: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/model_artifacts.json
Saved metadata: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_run_metadata.json


## 10. Output Summary

In [10]:
class_counts = pd.Series(class_map.reshape(-1)).value_counts().sort_index()
class_counts = class_counts.rename_axis("classvalue").reset_index(name="pixel_count")
class_counts_path = OUTPUT_DIR / f"{best_model_name}_class_pixel_counts.csv"
class_counts.to_csv(class_counts_path, index=False)

display(class_counts)
print("Saved class counts:", class_counts_path)
print("Output files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)

,classvalue,pixel_count
0,1,34506
1,2,153760


Saved class counts: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_class_pixel_counts.csv
Output files:
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/classification_reports.json
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/cnn_1d_model.keras
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/cnn_1d_scaler.joblib
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/cnn_1d_training_history.csv
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/model_artifacts.json
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/model_metrics.csv
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_class_map.tif
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_class_pixel_counts.csv
- /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_local/random_forest_confusion_matrix.csv
- /mnt